# **TF-IDF and Cosine Similarity**
-> ini bakal masuk ke Quiz kedua yagesyak

*TF-IDF (Term Frequency - Inverse Document Frequency)* => untuk ngecek seberapa penting suatu kata dalam dokumen (akan ada cara menual dan pake library)  
*Cosine Similarity* => ngeliat seberapa mirip dokumen satu dengan lainnya  

---

TF = jumlah kemunculan kata dalam dokumen / jumlah kata di dokumen  
IDF = log e (Jumlah dokumen / jumlah dokumen dengan suatu kata) => ngecek seberapa unique katanya, kalo sering malah jadi kecil
TF-IDF = TF * IDF

In [38]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import pandas as pd
import math  # buat masukin log nya
import numpy as np  # utk cosine similarity (perkalian vektor)
from numpy.linalg import norm  # utk ambil panjang dari vektor

In [39]:
eng_stopwords = stopwords.words('english')
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in eng_stopwords]
    return tokens

In [40]:
docs = ["Sally sells shiny silver seashells by the calm seashore every sunny Sunday", 
        "Sally stacks slippery silver seashells by the windy seashore each cloudy Sunday", 
        "Sally studies silver coins near the seashore while analyzing ancient economic history in a quiet museum"]
processed_docs = [preprocess(doc) for doc in docs]
print(processed_docs)

[['sally', 'sells', 'shiny', 'silver', 'seashells', 'calm', 'seashore', 'every', 'sunny', 'sunday'], ['sally', 'stacks', 'slippery', 'silver', 'seashells', 'windy', 'seashore', 'cloudy', 'sunday'], ['sally', 'studies', 'silver', 'coins', 'near', 'seashore', 'analyzing', 'ancient', 'economic', 'history', 'quiet', 'museum']]


In [41]:
# Create unique vocab

# vocab = set()
# for doc in processed_docs:
#     for word in doc:
#         vocab.add(word)        
# vocab = sorted(vocab)

vocab = sorted(set(word for doc in processed_docs for word in doc))
print(vocab)

['analyzing', 'ancient', 'calm', 'cloudy', 'coins', 'economic', 'every', 'history', 'museum', 'near', 'quiet', 'sally', 'seashells', 'seashore', 'sells', 'shiny', 'silver', 'slippery', 'stacks', 'studies', 'sunday', 'sunny', 'windy']


In [42]:
# Word Dictionaries
word_dicts = []
for doc in processed_docs:
    word_dict = dict.fromkeys(vocab, 0)
    
    for word in doc:
        if word in word_dict:
            word_dict[word] += 1
    word_dicts.append(word_dict)
    
pd.DataFrame(word_dicts)

,analyzing,ancient,calm,cloudy,coins,economic,every,history,museum,near,...,seashore,sells,shiny,silver,slippery,stacks,studies,sunday,sunny,windy
0,0,0,1,0,0,0,1,0,0,0,...,1,1,1,1,0,0,0,1,1,0
1,0,0,0,1,0,0,0,0,0,0,...,1,0,0,1,1,1,0,1,0,1
2,1,1,0,0,1,1,0,1,1,1,...,1,0,0,1,0,0,1,0,0,0


In [45]:
# Calculate TF (Term Frequency)
def compute_TF(word_dict):
    tf_dict = {}
    total_words = sum(word_dict.values())
    
    for word, count in word_dict.items():
        tf_dict[word] = count / total_words
        
    return tf_dict

In [46]:
tf_dicts = [compute_TF(word_dict) for word_dict in word_dicts]
pd.DataFrame(tf_dicts)

,analyzing,ancient,calm,cloudy,coins,economic,every,history,museum,near,...,seashore,sells,shiny,silver,slippery,stacks,studies,sunday,sunny,windy
0,0.000000,0.000000,0.1,0.000000,0.000000,0.000000,0.1,0.000000,0.000000,0.000000,...,0.100000,0.1,0.1,0.100000,0.000000,0.000000,0.000000,0.100000,0.1,0.000000
1,0.000000,0.000000,0.0,0.111111,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.111111,0.0,0.0,0.111111,0.111111,0.111111,0.000000,0.111111,0.0,0.111111
2,0.083333,0.083333,0.0,0.000000,0.083333,0.083333,0.0,0.083333,0.083333,0.083333,...,0.083333,0.0,0.0,0.083333,0.000000,0.000000,0.083333,0.000000,0.0,0.000000


In [59]:
# Calculate IDF (Inverse Document Frequency)
def compute_IDF(word_dicts):
    idf_dict = {}
    N = len(word_dicts)
    
    # Jumlah dokumen dengan t di dalamnya
    for word_dict in word_dicts:
        for word, count in word_dict.items():
            if count > 0:
                idf_dict[word] = idf_dict.get(word, 0) + 1  # Kalo baru, bakal diset ke 0
                
    for word, df in idf_dict.items():
        idf_dict[word] = math.log10(N / df)
    
    return idf_dict

IDF Dict Explanation
"apple peach orange"  
"apple peach apple"  
"apple orange"  
"apple cherry"  
\
idf_dict['apple'] = (df = 4) log(4/4) -> 0

In [60]:
idf_dict = compute_IDF(word_dicts)
pd.DataFrame(list(idf_dict.items()), columns=['Word', 'IDF'])

,Word,IDF
0,calm,0.477121
1,every,0.477121
2,sally,0.000000
3,seashells,0.176091
4,seashore,0.000000
5,sells,0.477121
6,shiny,0.477121
7,silver,0.000000
8,sunday,0.176091
9,sunny,0.477121


In [62]:
def computer_TFIDF(tf_dict, idf_dict):
    tfidf_dict = {}
    
    for word, tf in tf_dict.items():
        tfidf_dict[word] = tf * idf_dict.get(word, 0)
        
    return tfidf_dict

In [63]:
tfidf_dicts = [computer_TFIDF(tf_dict, idf_dict) for tf_dict in tf_dicts]

pd.DataFrame(tfidf_dicts)

,analyzing,ancient,calm,cloudy,coins,economic,every,history,museum,near,...,seashore,sells,shiny,silver,slippery,stacks,studies,sunday,sunny,windy
0,0.00000,0.00000,0.047712,0.000000,0.00000,0.00000,0.047712,0.00000,0.00000,0.00000,...,0.0,0.047712,0.047712,0.0,0.000000,0.000000,0.00000,0.017609,0.047712,0.000000
1,0.00000,0.00000,0.000000,0.053013,0.00000,0.00000,0.000000,0.00000,0.00000,0.00000,...,0.0,0.000000,0.000000,0.0,0.053013,0.053013,0.00000,0.019566,0.000000,0.053013
2,0.03976,0.03976,0.000000,0.000000,0.03976,0.03976,0.000000,0.03976,0.03976,0.03976,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.03976,0.000000,0.000000,0.000000


### Library

In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [72]:
vectorizer = TfidfVectorizer()
analyze = vectorizer.build_analyzer()  # handles preprocessing dan tokenizationt

X = vectorizer.fit_transform(docs).toarray()
print(vectorizer.get_feature_names_out())

for i, doc in enumerate(docs):
    print(f"D{i+1}: {analyze(doc)}")
    print(f"X{i+1}: {X}")

['analyzing' 'ancient' 'by' 'calm' 'cloudy' 'coins' 'each' 'economic'
 'every' 'history' 'in' 'museum' 'near' 'quiet' 'sally' 'seashells'
 'seashore' 'sells' 'shiny' 'silver' 'slippery' 'stacks' 'studies'
 'sunday' 'sunny' 'the' 'while' 'windy']
D1: ['sally', 'sells', 'shiny', 'silver', 'seashells', 'by', 'the', 'calm', 'seashore', 'every', 'sunny', 'sunday']
X1: [[0.         0.         0.2667197  0.35070436 0.         0.
  0.         0.         0.35070436 0.         0.         0.
  0.         0.         0.20713164 0.2667197  0.20713164 0.35070436
  0.35070436 0.20713164 0.         0.         0.         0.2667197
  0.35070436 0.20713164 0.         0.        ]
 [0.         0.         0.2667197  0.         0.35070436 0.
  0.35070436 0.         0.         0.         0.         0.
  0.         0.         0.20713164 0.2667197  0.20713164 0.
  0.         0.20713164 0.35070436 0.35070436 0.         0.2667197
  0.         0.20713164 0.         0.35070436]
 [0.28403464 0.28403464 0.         0. 

### Cosine Similarity

In [80]:
for i in range(len(tfidf_dicts)):
    for j in range(i+1, len(tfidf_dicts)):
        vec1 = np.array([tfidf_dicts[i][word] for word in vocab])
        vec2 = np.array([tfidf_dicts[j][word] for word in vocab])
        sim_manual = np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))
        print(f"[Manual] Similarity D{i+1} vs D{j+1}: {sim_manual}")
        
        sim_library = np.dot(X[i], X[j]) / (norm(X[i]) * norm(X[j]))
        print(f"[Library] Similarity D{i+1} vs D{j+1}: {sim_library}")
        print()

[Manual] Similarity D1 vs D2: 0.057399164691234454
[Library] Similarity D1 vs D2: 0.38503226983446404

[Manual] Similarity D1 vs D3: 0.0
[Library] Similarity D1 vs D3: 0.13898983543255208

[Manual] Similarity D2 vs D3: 0.0
[Library] Similarity D2 vs D3: 0.13898983543255206

